# T-condition behavioural screen (Qwen2.5-VL-3B)Plan: `docs/plan_v2.md` section 4, step 3 -- the cheapest kill-test. **Does anincongruent note move the model's answer on >= ~50 objects (D41)?**Thin wrapper: all logic lives in `src/circuit_overlap/`. **[FILL IN]** cells needinformation the repo does not have; **[VERIFY]** cells check an assumption beforeit can silently corrupt the run.Dataset: `mgolov/Visual-Counterfact`, `split="color"`, 493 rows. Note its realcolumns -- `object`, `correct_answer` (a *stringified list* of acceptablecolours), `incorrect_answer`, `original_image`, `counterfact_image`.Settings: GPU T4, Internet ON.

## 1. Clone the repo and install

In [ ]:
# [FILL IN] your repo URL (or upload the repo as a Kaggle dataset instead)REPO_URL = ""REPO_DIR = "/kaggle/working/research_paper"import os, subprocess, sysif REPO_URL and not os.path.exists(REPO_DIR):    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)sys.path.insert(0, f"{REPO_DIR}/src")print(subprocess.run(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"],                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate datasetsimport torch, transformersprint(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 2. Build the object manifestThe T condition pastes notes on the **original_image** (real colour), and thenote then claims `incorrect_answer`.`correct_answer` is a stringified list such as `"['brown', 'white']"` -- severalcolours can be correct for one object. `parse_color_list` normalises it, and thescorer treats "correct" as membership in that set. `object` is **not** assumedunique, so ids get a row suffix while the prompt keeps the bare noun.

In [ ]:
import json, pathlibfrom datasets import load_datasetfrom circuit_overlap.data import parse_color_listOUT = pathlib.Path("/kaggle/working/vc_color"); (OUT / "images").mkdir(parents=True, exist_ok=True)EXCLUDE = {"radish", "spider"}          # as Lietzow do (plan 3.1)ds = load_dataset("mgolov/Visual-Counterfact", split="color")print(ds.column_names, len(ds))items, skipped = [], []for i, row in enumerate(ds):    name = str(row["object"]).strip().lower()    if name in EXCLUDE:        continue    try:        correct = parse_color_list(row["correct_answer"])        attack = parse_color_list(row["incorrect_answer"])[0]    except (ValueError, TypeError) as exc:        skipped.append((i, name, repr(exc))); continue    if attack in correct:         # no conflict to measure        skipped.append((i, name, "counterfactual colour is also correct")); continue    img_path = OUT / "images" / f"{name}_{i:04d}.png"    row["original_image"].convert("RGB").save(img_path)    items.append({"object_id": f"{name}_{i:04d}", "object_name": name,                  "image": str(img_path), "original_colors": list(correct),                  "counterfactual_color": attack})MANIFEST = OUT / "manifest.json"MANIFEST.write_text(json.dumps(items, indent=1))print(f"{len(items)} objects written, {len(skipped)} skipped")for s in skipped[:10]: print("  skipped:", s)

In [ ]:
from circuit_overlap.data import load_manifest, collect_color_vocabulary, check_colors_in_vocabularyentries = load_manifest(MANIFEST)vocab_from_data = collect_color_vocabulary(entries)notes_cfg_raw = json.loads(open(f"{REPO_DIR}/configs/notes.json").read())print(len(entries), "objects")print("colours in the data:", vocab_from_data)print("multi-colour objects:", sum(len(e.original_colors) > 1 for e in entries))print("not in configs/notes.json placeholder vocabulary:",      check_colors_in_vocabulary(entries, notes_cfg_raw["color_vocabulary"]))# run_screen unions these automatically, but paste vocab_from_data into# configs/notes.json so the committed config matches the data.

## 3. [VERIFY] Look at one attack image before spending GPU time

In [ ]:
from circuit_overlap.notes import render_note, variant_texts, matched_neutral_wordfrom circuit_overlap.tokens import mask_to_token_indices, token_grid_shapefrom circuit_overlap.t_screen import load_notes_configcfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")e = entries[0]img = e.open_image()print(e.object_name, "| correct:", e.original_colors, "| note says:", e.counterfactual_color)texts = variant_texts(e.primary_original_color, e.counterfactual_color,                      matched_neutral_word(e.counterfactual_color, cfg["neutral_words"]))for spec in cfg["specs"]:    r = render_note(img, texts["incongruent"], spec)    print(spec.name, "area", round(r.area_frac, 3), "font", r.font_name, r.font_size_px)    display(r.image.resize((320, 320)))

In [ ]:
# [VERIFY] mask -> image-token mapping against the processor's real grid.from transformers import AutoProcessorfrom circuit_overlap.t_screen import ScreenConfigconfig = ScreenConfig()        # [VERIFY] checkpoint id (STATE.md blocker 2)processor = AutoProcessor.from_pretrained(config.checkpoint,                                          min_pixels=config.pixels, max_pixels=config.pixels)enc = processor(text=["x"], images=[r.image], return_tensors="pt")print("processor grid_thw (t, h, w in patches):", enc["image_grid_thw"])print("our merged token grid for that image:", token_grid_shape(*r.image.size[::-1]))# h/2 and w/2 from the processor must equal our (rows, cols) before any H3 claim.

## 4. [VERIFY] Colour words must be single, distinct first tokens (D26)

In [ ]:
from circuit_overlap.t_screen import single_first_token_idsids, bad = single_first_token_ids(processor.tokenizer, vocab_from_data)print("ids:", ids)print("violating D26:", bad)# Any word here makes Delta undefined for the objects that use it: drop those# objects (and say so in RESULTS.md), do not quietly keep them.

## 5. Smoke run on 5 objects, then the full screen

In [ ]:
from circuit_overlap.t_screen import load_model, run_screenmodel, processor = load_model(config)records, summary = run_screen(entries[:5], model, processor, config, cfg)for r in records[:6]:    print(f"{r.layout:13s} {r.object_name:12s} correct={r.original_colors} "          f"ref={r.reference_color} | neutral: {r.answer_neutral!r} "          f"| attacked: {r.answer_incongruent!r} "          f"| shift {(r.delta_neutral - r.delta_incongruent):+.2f}")

In [ ]:
import timet0 = time.time()records, summary = run_screen(entries, model, processor, config, cfg)print(f"{time.time()-t0:.0f}s for {2*len(records)} generations"){k: v for k, v in summary.items() if k not in ("usable_object_ids", "color_vocabulary")}

## 6. Save the result JSON (download it and commit it to `results/`)

In [ ]:
from dataclasses import asdictfrom circuit_overlap.provenance import result_envelope, write_resultenvelope = result_envelope(    experiment="t_behavioural_screen",    payload={"summary": summary, "records": [asdict(r) for r in records]},    model="Qwen2.5-VL-3B",    checkpoint=config.checkpoint,    seed=config.seed,    prompt=config.prompt_template,    n=summary["n_objects"],    repo_dir=REPO_DIR,    dataset={"source_id": "mgolov/Visual-Counterfact", "split": "color",             "n_rows_total": len(ds), "n_after_exclusions": len(entries),             "excluded": sorted(EXCLUDE)},    layouts=[s.name for s in cfg["specs"]],    dtype=config.dtype,    pixels=config.pixels,)path = write_result("/kaggle/working/t_screen_qwen25vl3b.json", envelope)print(path, summary["n_objects_usable"], "usable objects; gate_passed =", summary["gate_passed"])

## 7. What the outcome means (plan section 4)| Result | Next ||---|---|| `gate_passed = True` (>= 50 usable objects) | Proceed to T-circuit discovery || Few flips but `continuous_fallback.n_objects_shifted` healthy | Use the continuous variant: patch on logit-difference shifts without an argmax flip || Few flips and little shift | Raise note salience (size, contrast) in `configs/notes.json`, or try PaliGemma-3B as primary |Then record the number in `docs/RESULTS.md` with model, n, seed, prompt andcommit, and log the 493-row / Lietzow-467 discrepancy in `docs/STATE.md`.